In [44]:
import re
import json
from ast import literal_eval
from typing import Tuple

def extract_xml(text: str, tag: str) -> str:
    """
    Extracts the content of the specified XML tag from the given text. Used for parsing structured responses

    Args:
        text (str): The text containing the XML.
        tag (str): The XML tag to extract content from.

    Returns:
        str: The content of the specified XML tag, or an empty string if the tag is not found.
    """
    match = re.search(f"<{tag}>(.*?)</{tag}>", text, re.DOTALL)
    return match.group(1) if match else ""


def parse_parameter(parameter: str) -> dict:
    """
    Parser the parameter string into a dictionary.
    key is the name of the parameter, value is the value of the parameter.
    parameter is like:
    'point: [426, 270]' -> {'point': [426, 270]}
    'region: [20, 300, 400, 500]' -> {'region': [20, 300, 400, 500]}
    'direction: up' -> {'direction': 'up'}
    'text: Click to manage account information.' -> {'text': 'Click to manage account information.'}
    """
    try:
        if ":" in parameter:
            parameter = parameter.strip().split(":", 1)
            key = parameter[0].strip()
            if key in ['point', 'region']:
                value = literal_eval(parameter[1].strip())
            else:
                value = parameter[1].strip()
            return {key: value}
        else:
            return {}
    except Exception as e:
        print(f"Error parsing parameter: {e}")
        print(f"Parameter: {parameter}")
        return {}


def extract_action(text: str) -> str:
    """
    Extract the ground truth action from the anwser.

    <answer>
        <action description>
            Click on the ['My account\nManage account info'] to (Click to manage account information.)
        </action description>
        <action>
            <name>
                TAP
            </name>
            <parameters>
                <parameter>
                    point: [426, 270]
                </parameter>
            </parameters>
        </action>
        <active region>
            region: [20, 300, 400, 500]
        </active region>
    </answer>

    a well defined action contains: name, parameters, active_region
    """
    answer = extract_xml(text, "answer").strip()
    action = extract_xml(answer, "action").strip()
    action_name = extract_xml(action, "name").strip()
    parameter = parse_parameter(
        extract_xml(action, "parameter").strip()
    )  # TODO suppose only one parameter
    active_region = parse_parameter(extract_xml(answer, "active region").strip())
    return action_name, parameter, active_region


def point_in_region(point: Tuple[int, int], region: Tuple[int, int, int, int]) -> bool:
    """
    Check if the point is in the region.
    """
    return region[0] <= point[0] <= region[2] and region[1] <= point[1] <= region[3]


def eval_action(gt_action, gt_parameter, gt_active_region, action, parameter) -> float:
    """
    Evaluate the action is correct and return the reward.
    Action Space includes:
        TAP, SWIPE, TYPE, with parameters: point, direction, text
        TASK_COMPLETE, PRESS_ENTER, TASK_IMPOSSIBLE, PRESS_BACK, PRESS_HOME, WAIT, with no parameters
    """

    def lcs(s1, s2):
        """
        Calculate the longest common subsequence (LCS) of two strings.
        """
        m, n = len(s1), len(s2)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if s1[i - 1] == s2[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1] + 1
                else:
                    dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
        return dp[m][n]

    reward = 0.0

    if gt_action != action:
        return reward # 动作不一致，奖励为0
    
    reward += 0.5 # 动作一致，奖励0.5
    
    if gt_action == "TAP":
        point = parameter.get("point", None)
        if (
            isinstance(point, list)
            and isinstance(gt_active_region, list)
        ):
            reward += 0.5 # 参数类型正确，奖励0.5
            if point_in_region(point, gt_active_region):
                reward += 1 # 点在区域内，奖励0.5
    
    elif gt_action == "SWIPE":
        direction = parameter.get("direction", None)
        gt_direction = gt_parameter.get("direction", None)
        if direction is not None and direction in ["up", "down", "left", "right"]:
            reward += 0.5 # 参数类型正确，奖励0.5
            if direction == gt_direction:
                reward += 0.5 # 方向一致，奖励0.5

    elif gt_action == "TYPE":
        text = parameter.get("text", None)
        gt_text = gt_parameter.get("text", None)
        if text is not None:
            reward += 0.5 # 参数类型正确，奖励0.5
            # 比较内容是否相似，lcs > 50% 则认为相似
            if lcs(text, gt_text) / min(len(text), len(gt_text)) > 0.5:
                reward += 0.5 # 内容相似，奖励0.5

    elif gt_action == "TASK_COMPLETE":
        if action == "TASK_COMPLETE" and parameter == {}:
            reward += 0.5 # 动作一致，奖励0.5
    
    elif gt_action == "PRESS_ENTER":
        if action == "PRESS_ENTER" and parameter == {}:
            reward += 0.5 # 动作一致，奖励0.5
        
    elif gt_action == "TASK_IMPOSSIBLE":
        if action == "TASK_IMPOSSIBLE" and parameter == {}:
            reward += 0.5 # 动作一致，奖励0.5
        
    elif gt_action == "PRESS_BACK":
        if action == "PRESS_BACK" and parameter == {}:
            reward += 0.5 # 动作一致，奖励0.5
        
    elif gt_action == "PRESS_HOME":
        if action == "PRESS_HOME" and parameter == {}:
            reward += 0.5 # 动作一致，奖励0.5
        
    elif gt_action == "WAIT":
        if action == "WAIT" and parameter == {}:
            reward += 0.5 # 动作一致，奖励0.5
    else:
        reward += 0.0 # 动作不一致，奖励为0

    return reward



In [48]:
generation_1 = """<think>
The history indicates that 'Done Button, Done' was already clicked, suggesting that the initial setup or input related to car rental details in Paris had been completed. Now, to proceed with finding a rental, the next logical step is to enter the pickup location, which is CDG (Paris-Orly Airport).
</think>
<answer>
        <action description>
            Tap on the text field related to 'Pickup location' to begin entering the desired location and time.
        </action description>
        <action>
            <name>
                TASK_COMPLETE
            </name>
            <parameters>
                <parameter>
                    text: CDG
                </parameter>
            </parameters>
        </action>
</answer>
"""
gt_1 = """<answer>
        <action description>
            Type CDG
        </action description>
        <action>
            <name>
                TYPE
            </name>
            <parameters>
                <parameter>
                    text: CDG
                </parameter>
            </parameters>
        </action>
    </answer>
"""
generation_2 = """<think> The task is to open the Podcast Player and check the "Downloads" tab to play the third downloaded podcast episode. To achieve this, I need to open the Podcast Player app first. The Play Store icon is visible on the screen, which is the entry point to the Podcast Player app. Therefore, my task is to open the Play Store app. </think>
<answer>
<action description>
    I will tap on the Play Store icon to open the app.
</action>
<action>
    <name>
        TAP
    </name>
    <parameters>
        <parameter>
            point: [18, 85]
        </parameter>
    </parameters>
</action>
</answer>
"""
gt_2 = """<answer>
        <action description>
            Click on the ['Podcast Player', 'Podcast Player']
        </action description>
        <action>
            <name>
                TAP
            </name>
            <parameters>
                <parameter>
                    point: [169, 386]
                </parameter>
            </parameters>
        </action>
        <active region>
            region: [52, 335, 246, 451]
        </active region>
    </answer>"""

In [49]:
def test_action_reward(generation, gt):
    gt_action, gt_parameter, gt_active_region = extract_action(gt)
    action, parameter, _ = extract_action(generation)
    print("gt action: ", gt_action, "gt parameter: ", gt_parameter, "gt active region: ", gt_active_region)
    print("generation action: ", action, "generation parameter: ", parameter)
    if gt_action:
        # gt 有可解析的action
        reward = eval_action(
            gt_action, gt_parameter, gt_active_region, action, parameter
        )
        print(reward)
    else:
        # gt 没有可解析的action
        print("gt 没有可解析的action")


In [50]:
test_action_reward(generation_1, gt_1)
test_action_reward(generation_2, gt_2)

gt action:  TYPE gt parameter:  {'text': 'CDG'} gt active region:  {}
generation action:  TASK_COMPLETE generation parameter:  {'text': 'CDG'}
0.0
gt action:  TAP gt parameter:  {'point': [169, 386]} gt active region:  {'region': [52, 335, 246, 451]}
generation action:  TAP generation parameter:  {'point': [18, 85]}
0.5


In [51]:
parse_parameter("point: [18, 85]")

{'point': [18, 85]}

In [52]:
print(parse_parameter(extract_xml(generation_1, "parameter").strip()))

{'text': 'CDG'}


In [43]:
extract_xml(generation_1, "parameter").strip()

'text: CDG'